# KWS eval part 1: Label posteriors with ZIPA
In this notebook I've given you several pieces of the puzzle for how to get KWS probability for Tira audio using the ZIPA model.
I've broken it down into 3 stages:
1. Loading KWS sentences and audio
2. Computing logits with ZIPA
3. Getting label posteriors for each sentence using CTC loss

We don't quite get to actual keyword hit probability in this exercise, but we'll get there next.

## Objective
As you go through this exercise, rather than just fill out the notebook, I'm asking you to create a python script (I've left a blank file in `scripts/eval/kws_ctc_eval.py`) that will get keyword probabilities for every sentence in the dataset.
Think about the components I've given you here, and how they can be arranged in a logical sequence so that you can write a single script that when run end-to-end will perform all the steps needed to evaluate the ZIPA model on our Tira KWS dataset.

## Next steps
After you finish this task the next objective will be to swap from naive label probability to keyword hit probability, and then to use the probabilities to compute evaluation metrics like mAP, F1 and AUROC.

In [1]:
from tira_kws.dataloading import load_capstone_kws_cuts, get_k2_dataloader
from tira_kws.models.zipa import load_zipa_small_crctc
from tira_kws.constants import CAPSTONE_SUPERVISIONS, CAPSTONE_KEYWORDS
from lhotse import SupervisionSet
import pandas as pd
import torch

/home/obandopadhyay/Projects/zipa/zipformer_crctc/scaling.py:1306: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, x, y, alpha):
/home/obandopadhyay/Projects/zipa/zipformer_crctc/scaling.py:1315: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, ans_grad):
/home/obandopadhyay/Projects/zipa/zipformer_crctc/scaling.py:1512: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(
/home/obandopadhyay/Projects/zipa/zipformer_crctc/scaling.py:1551: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, ans_grad: Tensor):


## Prior reading
Please read this [HuggingFace chapter](https://huggingface.co/learn/llm-course/en/chapter3/4) and [PyTorch tutorial](https://sebastianraschka.com/teaching/pytorch-1h/_) before trying to understand any of the code here, and use them as references if anything here is confusing.

## Keywords
Keywords are stored in a CSV file.
We can load this with Pandas.

In [2]:
df = pd.read_csv(CAPSTONE_KEYWORDS)
df.head()

,keyword,keyword_id,gloss
0,àpɾí,0.0,boy
1,ðɔ̀mɔ̀cɔ̀,1.0,man
2,lɛ́ŋgɛ́n,2.0,mother
3,ŋɔ̀ɽíŋgɔ́,3.0,donkey
4,mùðù,4.0,leopard


We'll save the list of keyword strings for easy reference later.

In [3]:
keywords = df['keyword'].tolist()
keywords

['àpɾí',
 'ðɔ̀mɔ̀cɔ̀',
 'lɛ́ŋgɛ́n',
 'ŋɔ̀ɽíŋgɔ́',
 'mùðù',
 'ùɾnɔ̀',
 'ðàŋàl',
 'kə̀və̀lɛ̀ðɔ́',
 'kàŋú',
 'íŋgá_nɔ̀nà']

## Sentence data
KWS sentence metadata are stored in JSONL (= Javascript Object Notation Lines), where every line is a single JSON object which is written similarly to a Python dictionary and can be read as a Python dictionary directly. Let's print the first two lines of the JSONL file using the `json` library for formatting.

In [4]:
import json
with open(CAPSTONE_SUPERVISIONS) as f:
    lines = f.readlines()[:1]
json_objects = [json.loads(line) for line in lines]
json_objects_formatted = [
    json.dumps(obj, indent=2, ensure_ascii=False)
    for obj in json_objects
]
jsonl_header = "\n".join(json_objects_formatted)
print(jsonl_header)

{
  "id": "àpɾí_484_positive",
  "start": 1969.12,
  "duration": 2.851,
  "channel": 0,
  "supervisions": [
    {
      "id": "àpɾí_484_positive",
      "recording_id": "HH02122021",
      "start": 0.0,
      "duration": 2.851,
      "channel": 0,
      "text": "àpɾí jǎŋîvlɛ̀ðɔ́ ǹdòbàgɛ̀",
      "language": "tic",
      "speaker": "Himidan",
      "custom": {
        "fst_text": "àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀",
        "gloss": "boy[NOM,SG] aux pull[CLJ,VENT,IPFV,1SG.OBJ,aux] tomorrow[]",
        "root": "àpɾí ŋgá vəlɛð nd̪ɔ̀bàgɛ̀",
        "translation": "The boy will pull me here tomorrow.",
        "keyword": "àpɾí",
        "norm_text": "àpɾí jáŋɛ̂_və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀",
        "record_type": "positive"
      }
    }
  ],
  "recording": {
    "id": "HH02122021",
    "sources": [
      {
        "type": "file",
        "channels": [
          0
        ],
        "source": "/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH02122021.wav"
      

Rather than loading JSONL metadata manually, we use the Lhotse library which is designed for managing speech datasets.
The function `tira_kws/dataloading.py#load_capstone_kws_cuts()` uses Lhotse to load KWS data into a `CutSet` object (i.e. a set of audio snippets cut out from the source audio).
As we can see, the cuts contain the same data as the JSONL file above.

Read the [Lhotse documentation](https://lhotse.readthedocs.io/en/latest/cuts.html) for more information on `Cut` and `CutSet` objects.
By default Lhotse returns a generator rather than a list.
To load cuts as a list instead (so we can index them), call `cuts.to_eager()`.

In [5]:
cuts = load_capstone_kws_cuts()
cuts = cuts.trim_to_supervisions()
cuts = cuts.to_eager()
cuts[0]

MonoCut(id='àpɾí_484_positive', start=1969.12, duration=2.851, channel=0, supervisions=[SupervisionSegment(id='àpɾí_484_positive', recording_id='HH02122021', start=0.0, duration=2.851, channel=0, text='àpɾí jǎŋîvlɛ̀ðɔ́ ǹdòbàgɛ̀', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'gloss': 'boy[NOM,SG] aux pull[CLJ,VENT,IPFV,1SG.OBJ,aux] tomorrow[]', 'root': 'àpɾí ŋgá vəlɛð nd̪ɔ̀bàgɛ̀', 'translation': 'The boy will pull me here tomorrow.', 'keyword': 'àpɾí', 'norm_text': 'àpɾí jáŋɛ̂_və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'record_type': 'positive'}, alignment=None)], features=None, recording=Recording(id='HH02122021', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH02122021.wav')], sampling_rate=16000, num_samples=55650022, duration=3478.126375, channel_ids=[0], transforms=None), custom={'fst_text': 'àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'gloss': 'boy[NOM,SG]

In [6]:
len(cuts)

300

## Data filtering
It's useful to load all sentences for a particular keyword rather than all sentences at once.

In [7]:
keyword = keywords[1]
keyword_cuts = cuts.filter(
    lambda cut: cut.custom.get('keyword', None) == keyword
)
keyword_cuts = keyword_cuts.to_eager()
keyword, keyword_cuts

('ðɔ̀mɔ̀cɔ̀', CutSet(len=20) [underlying data type: <class 'list'>])

We can also load only the negative records from the dataset.

In [8]:
negative_cuts = cuts.filter(
    lambda cut: cut.custom.get('record_type', None) == 'negative'
)
negative_cuts = negative_cuts.to_eager()
negative_cuts

CutSet(len=100) [underlying data type: <class 'list'>]

In [9]:
# let's work on 'ðɔ̀mɔ̀cɔ̀' and all the negative words first_cuts
kw_1_cutset = keyword_cuts + negative_cuts
kw_1_cutset

CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>]

In [10]:
kw_1_cutset[21]

MonoCut(id='13375_negative', start=103.26, duration=1.55, channel=0, supervisions=[SupervisionSegment(id='13375_negative', recording_id='HH20230531-2', start=0.0, duration=1.55, channel=0, text='ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò', 'gloss': 'slice[CLÐ,INF] meat/vegetable/prey[NOM,PL,left_h] PARTICLE[] good[CLÐ]', 'root': 'ð r̀ðɛ̀ ánó icəlo', 'translation': 'To cut meat is good', 'norm_text': 'ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò', 'record_type': 'negative'}, alignment=None)], features=None, recording=Recording(id='HH20230531-2', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH20230531-2.wav')], sampling_rate=16000, num_samples=28642627, duration=1790.1641875, channel_ids=[0], transforms=None), custom={'fst_text': 'ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò', 'gloss': 'slice[CLÐ,INF] meat/vegetable/prey[NOM,PL,left_h] PARTICLE[] good[CLÐ]', 'roo

#### Get relevant cutset for each keyword, put all cutsets into a list
The code we need before this that doesn't need to be a function is getting the cuts, and getting the negative cuts

In [11]:
def get_keyword_cutset(keyword, cuts):
    # get cuts for the keyword
    keyword_cuts = cuts.filter(
        lambda cut: cut.custom.get('keyword', None) == keyword
    )
    keyword_cuts = keyword_cuts.to_eager()

    # concatenate with negative cuts
    kw_cutset = keyword_cuts + negative_cuts

    return kw_cutset

In [12]:
def get_all_kw_cutsets(keyword_list, cuts):
    kw_cutsets = {}

    for kw in keyword_list:
        kw_cutset = get_keyword_cutset(kw, cuts)
        kw_cutsets[kw] = kw_cutset

    return kw_cutsets

In [13]:
kw_cutsets = get_all_kw_cutsets(keywords, cuts)
kw_cutsets

{'àpɾí': CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>],
 'ðɔ̀mɔ̀cɔ̀': CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>],
 'lɛ́ŋgɛ́n': CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>],
 'ŋɔ̀ɽíŋgɔ́': CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>],
 'mùðù': CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>],
 'ùɾnɔ̀': CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>],
 'ðàŋàl': CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>],
 'kə̀və̀lɛ̀ðɔ́': CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>],
 'kàŋú': CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>],
 'íŋgá_nɔ̀nà': CutSet(len=120) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>]}

Currently, it makes sense that we'll have 23 records in each cut, because it's 20 for each keyword (10 positive, 10 close negative) and 3 negatives for all of them. When we have the updated dataset, each of the cutsets should be of length 120. Batching will make more sense then as well.

## Dataloading and batching
We use a dataloader to serve batches to the model.

In [14]:
dataloader = get_k2_dataloader(cuts)
dataloader

In [15]:
# lets use a dataloader, but upload our filtered kw_1 cuts
kw_1_dl = get_k2_dataloader(kw_1_cutset)
kw_1_dl

The dataloader batches the data into a roughly fixed size (no more than 32 records by default).
Each batch has 'inputs' (the audio features to be fed to the model, stored as torch tensors) and 'supervisions' (the metadata for each record in the batch).

See [Torch documentation](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) for more on dataloaders.

In [16]:
for i, batch in enumerate(dataloader):
    if i >= 5:
        break
    print(batch.keys())
    # input shape is batch_size, sequence_length, feature_dimension
    print(batch['inputs'].shape)

dict_keys(['inputs', 'supervisions'])
torch.Size([32, 231, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([32, 186, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([32, 155, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([32, 209, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([10, 727, 80])


In [17]:
# let's look at the batches for my keyword cutset
for batch in kw_1_dl:
    print(batch.keys())
    # input shape is batch_size, sequence_length, feature_dimension
    print(batch['inputs'].shape)

dict_keys(['inputs', 'supervisions'])
torch.Size([8, 329, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([17, 156, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([26, 131, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([15, 188, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([1, 557, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([13, 209, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([12, 223, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([11, 241, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([10, 289, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([7, 522, 80])


Let's look more closely at the metadata stored under "supervisions."

In [18]:
first_batch = next(iter(dataloader))
supervisions = first_batch['supervisions']
supervisions.keys()

dict_keys(['text', 'sequence_idx', 'start_frame', 'num_frames', 'cut'])

In [19]:
# looking at batch 1 for kw1
b1_kw1 = next(iter(kw_1_dl))
supervs_kw_1 = b1_kw1['supervisions']
supervs_kw_1.keys()

dict_keys(['text', 'sequence_idx', 'start_frame', 'num_frames', 'cut'])

All records in a batch are automatically padded to the same length, but they were not the same length originally.
Lhotse includes a 'num_frames' field that lets us know the original length of each record.

In [20]:
supervisions['num_frames']

tensor([231, 230, 228, 227, 227, 226, 225, 224, 224, 224, 224, 223, 223, 221,
        221, 219, 218, 218, 218, 217, 216, 215, 215, 215, 215, 214, 213, 212,
        211, 211, 210, 210], dtype=torch.int32)

In [21]:
supervs_kw_1['num_frames']

tensor([329, 324, 320, 310, 305, 305, 294, 293], dtype=torch.int32)

`batch['supervisions']['cut']` is a list of `lhotse.MonoCut` objects, one for each record in the batch.
We can access the record's metadata (transcription, keyword, whether it's positive or negative) from the `.custom` field of the `MonoCut` object.

In [22]:
print(len(supervisions['cut']))
print(supervisions['cut'][0])

32
MonoCut(id='kàŋú_11178_positive', start=1404.27, duration=2.31, channel=0, supervisions=[SupervisionSegment(id='kàŋú_11178_positive', recording_id='HH20220919', start=0.0, duration=2.31, channel=0, text='kúkù kàŋú t̪ólìɲà vɽàn', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'kúkù kàŋú t̪ólèɲá ávàn', 'gloss': '1st.born.male.child[NOM,SG] see[CLG,VENT,PFV] lion[ACC,SG,left_h] very.much[left_h]', 'root': 'kúkù aŋ t̪òlé àvàn', 'translation': 'even Kuku saw the lion OR Kuku even/also saw the lion', 'keyword': 'kàŋú', 'norm_text': 'kúkù kàŋú t̪ólèɲá vɽàn', 'record_type': 'positive'}, alignment=None)], features=None, recording=Recording(id='HH20220919', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH20220919.wav')], sampling_rate=16000, num_samples=38687938, duration=2417.996125, channel_ids=[0], transforms=None), custom={'fst_text': 'kúkù kàŋú t̪ólèɲá á

In [23]:
print(supervisions['cut'][0].custom)

{'fst_text': 'kúkù kàŋú t̪ólèɲá ávàn', 'gloss': '1st.born.male.child[NOM,SG] see[CLG,VENT,PFV] lion[ACC,SG,left_h] very.much[left_h]', 'root': 'kúkù aŋ t̪òlé àvàn', 'translation': 'even Kuku saw the lion OR Kuku even/also saw the lion', 'keyword': 'kàŋú', 'norm_text': 'kúkù kàŋú t̪ólèɲá vɽàn', 'record_type': 'positive', 'dataloading_info': {'rank': 0, 'world_size': 1, 'worker_id': None}}


In [24]:
# looking at the same for our batch of 2
print(len(supervs_kw_1['cut']))
print(supervs_kw_1['cut'][0])
print()
print(supervs_kw_1['cut'][0].custom)

8
MonoCut(id='ðɔ̀mɔ̀cɔ̀_15663_positive', start=25.3, duration=3.29, channel=0, supervisions=[SupervisionSegment(id='ðɔ̀mɔ̀cɔ̀_15663_positive', recording_id='HH20230929', start=0.0, duration=3.29, channel=0, text='ðə̀mɔ̀cɔ́ ðávə̀lɛ̀ðà ɔ̀ndi nd̪ɔ̀bà', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'ðɔ̀mɔ̀cɔ̀ ðə́və̀lɛ̀ðà ɔ̀ndì nd̪ɔ̀bà', 'gloss': 'man/male[NOM,SG] pull[CLÐ,VENT,DEP,left_h] what[] tomorrow[]', 'root': 'ðɔ̀mɔ̀cɔ̀ vəlɛð ɔ̀ndì nd̪ɔ̀bà', 'translation': 'What will the man pull tomorrow? ', 'keyword': 'ðɔ̀mɔ̀cɔ̀', 'norm_text': 'ðɔ̀mɔ̀cɔ̀ ðá_və́lɛ̀ðà ɔ̀ndì nd̪ɔ̀bà', 'record_type': 'positive'}, alignment=None)], features=None, recording=Recording(id='HH20230929', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH20230929.wav')], sampling_rate=16000, num_samples=45239216, duration=2827.451, channel_ids=[0], transforms=None), custom={'fst_text': 'ðɔ̀mɔ̀cɔ̀ ðə́və̀lɛ̀ðà ɔ̀ndì nd̪ɔ̀bà

#### Batch each of the keyword cutsets, loading them into a dictionary

In [25]:
def batch_cutset(cutset):
    batches = []
    batch = get_k2_dataloader(cutset)
    batches.append(batch)
    return batches

kw_1_batches = batch_cutset(kw_1_cutset)
kw_1_batches

## Model
I've sequestered some of the code from ZIPA in `tira_kws/models/zipa.py` which we can use to load some of the ZIPA models.
For now let's stick with Zipa-Cr-Small.
Don't forget to put the model to GPU after loading, otherwise inference will be extremely slow.

In [26]:
model = load_zipa_small_crctc()
model = model.to('cuda')
type(model)

model.AsrModel

Here's a sample code snippet for getting CTC logits for a batch from the dataloader.

In [27]:
first_batch = next(iter(dataloader))

with torch.no_grad(): # what does 'no_grad' mean and why are we using it here?
    inputs = first_batch['inputs'].to('cuda')
    input_lengths = first_batch['supervisions']['num_frames'].to('cuda')
    # why put inputs and input_lens to cuda?
    # Hint: try running the snippet without `.to('cuda')`

    # Embeddings = hidden representation prior to output layer
    embeds, output_lengths = model.forward_encoder(inputs, input_lengths)
    # CTC output = logits from final layer
    ctc_logits = model.ctc_output(embeds)
    

In [29]:
# get the ctc logits for our first batch

with torch.no_grad():
    inputs_b1 = b1_kw1['inputs'].to('cuda')
    input_lengths_b1 = b1_kw1['supervisions']['num_frames'].to('cuda')

    # Embeddings = hidden representation prior to output layer
    embeds_b1, output_lengths_b1 = model.forward_encoder(inputs_b1, input_lengths_b1)
    # CTC output = logits from final layer
    ctc_logits_b1 = model.ctc_output(embeds_b1)

ctc_logits_b1.shape

torch.Size([8, 161, 127])

#### Feed each of the batches into the model

In [49]:
# let's look at the batches for my keyword cutset
'''for batch in kw_1_dl:
    print(batch.keys())
    # input shape is batch_size, sequence_length, feature_dimension
    print(batch['inputs'].shape)'''

losses = []
sentence_infos = []
# iterate through each of the keyword cutsets
for kw in kw_cutsets:
    print(kw, kw_cutsets[kw])
    kw_cutset = kw_cutsets[kw]

    # tokenize the keyword
    kw_tokenized = tokenizer.encode(kw)
    print(kw_tokenized)

    # batch each of the cutsets
    dataloader = get_k2_dataloader(kw_cutset)
    for batch in dataloader:

        # feed each batch into the model and get the ctc logits
        # this will get us the ctc logits for all the batches
        with torch.no_grad():
            inputs = batch['inputs'].to('cuda')
            input_lengths = batch['supervisions']['num_frames'].to('cuda')
            embeds, output_lengths = model.forward_encoder(inputs, input_lengths)
            ctc_logits = model.ctc_output(embeds)
            
            print('CTC LOGITS: ', len(ctc_logits))

        # get the probabilities from the logits for each batch
        batch_size = ctc_logits.shape[0]
        targets = torch.tensor([kw_tokenized]*batch_size)
        target_lengths = torch.tensor([len(kw_tokenized)]*batch_size)
        ctc_probs = log_softmax(ctc_logits)

        loss = ctc_loss(
            ctc_probs.permute(1, 0, 2),
            targets,
            output_lengths,
            target_lengths,
        )
        print('LOSS:', loss, loss.shape)

        losses.append(loss)

        
        # get sentence info for each record by batch
        batch_sent_info = []
        monocut_list = batch['supervisions']['cut']
        
        for record in monocut_list:
            record_info = {}
            #sentence = record.custom['fst_text']
            record_info['sentence'] = record.custom['fst_text']
            record_info['record_type'] = record.custom['record_type']
            if 'keyword' in record.custom:
                record_info['keyword'] = record.custom['keyword']
            else:
                record_info['keyword'] = 0
            
            batch_sent_info.append(record_info)
        print('BATCH SENT INFO: ', batch_sent_info)

        sentence_infos.append(batch_sent_info)

        #probs = sigmoid(loss)
        #print('PROBS: ', probs)
losses, sentence_infos

àpɾí CutSet(len=23) [underlying data type: <class 'lhotse.lazy.LazyIteratorChain'>]
[3, 4, 2, 19, 80, 12, 2]
CTC LOGITS:  1
LOSS: tensor([-57.1939], device='cuda:0') torch.Size([1])
BATCH SENT INFO:  [{'sentence': 'àn ðúɽì ńdì ðáɾɔ́ðɛ́ ùrnɔ̀ ðàl t̪ùdí àɾðàt̪á àpɾíɲá', 'record_type': 'close_negative', 'keyword': 'àpɾí'}]
CTC LOGITS:  3
LOSS: tensor([83.8342, 97.8649, 73.7603], device='cuda:0') torch.Size([3])
BATCH SENT INFO:  [{'sentence': 'ŋɛ́n ŋɔ́ɽàt̪átɛ́ àpɾíɲá nɛ̀ lə̀və́r', 'record_type': 'close_negative', 'keyword': 'àpɾí'}, {'sentence': 'ðàŋàl ðá və́lɛ̀ðà àpɾíɲá', 'record_type': 'close_negative', 'keyword': 'àpɾí'}, {'sentence': 'ùrnɔ̀ káɾɔ́ àpɾí jájà ɔ́ndì', 'record_type': 'positive', 'keyword': 'àpɾí'}]
CTC LOGITS:  6
LOSS: tensor([138.9064, 114.4132,  96.3413,  70.1231,  49.7437,  58.0967],
       device='cuda:0') torch.Size([6])
BATCH SENT INFO:  [{'sentence': 'kúkù ká útìðìjɔ̀ àpɾíɲá lɛ́ŋgɛ́n', 'record_type': 'close_negat

([tensor([-57.1939], device='cuda:0'),
  tensor([83.8342, 97.8649, 73.7603], device='cuda:0'),
  tensor([138.9064, 114.4132,  96.3413,  70.1231,  49.7437,  58.0967],
         device='cuda:0'),
  tensor([99.0592, 86.3393, 98.3944], device='cuda:0'),
  tensor([ 76.3228, 130.0816,  84.6075], device='cuda:0'),
  tensor([115.2515, 156.6272, 120.5622], device='cuda:0'),
  tensor([63.5724, 64.4986], device='cuda:0'),
  tensor([98.6237, 66.1690], device='cuda:0'),
  tensor([66.7754, 36.0334], device='cuda:0'),
  tensor([38.5513, 52.3147, 46.5234], device='cuda:0'),
  tensor([118.9912,  82.9762,  55.9636,  50.4699,  44.2292,  49.4454],
         device='cuda:0'),
  tensor([72.3773, 54.9511, 60.1440], device='cuda:0'),
  tensor([86.8335, 70.1775, 75.5472], device='cuda:0'),
  tensor([41.1887, 25.4328], device='cuda:0'),
  tensor([26.0301, 39.3046], device='cuda:0'),
  tensor([38.7766, 49.4567], device='cuda:0'),
  tensor([-59.1028], device='cuda:0'),
  tensor([57.5276, 54.0894, 63.2461, 60.1717],

## Tokenization
Before we can get keyword probabilities, we need to encode our keywords as token ids.
We can do this using the ZIPA sentencepiece tokenizer.

In [31]:
import sentencepiece
from tira_kws.constants import ZIPA_SENTENCEPIECE_MODEL

tokenizer = sentencepiece.SentencePieceProcessor()
tokenizer.load(str(ZIPA_SENTENCEPIECE_MODEL))

True

In [32]:
tokenizer.encode(['àpɾí', 'jícə̀lò'])

[[3, 4, 2, 19, 80, 12, 2], [3, 13, 12, 2, 6, 50, 2, 15, 18, 2]]

In [33]:
# TODO: encode every keyword from the first batch
# YOUR CODE HERE
encoded_batch_labels = ...

In [34]:
# let's encode the keywords from our first batch, which should have 2 keywords
# it might be best to do this by identifying the relevant keyword from supervisions

print(b1_kw1['supervisions']['cut'][0])
print()
print(b1_kw1['supervisions']['cut'][1])
print()

print(len(b1_kw1['supervisions']['cut']))
print()

fts = b1_kw1['supervisions']['cut'][0].custom
#print((b1_kw1['supervisions']['cut'][0].custom)[keyword])
print(fts['keyword'])

for i in range(0, len(b1_kw1['supervisions']['cut'])):
    print(i)
    print((b1_kw1['supervisions']['cut'][i].custom)['keyword'])


MonoCut(id='ðɔ̀mɔ̀cɔ̀_3320_positive', start=134.088, duration=4.55, channel=0, supervisions=[SupervisionSegment(id='ðɔ̀mɔ̀cɔ̀_3320_positive', recording_id='HH20210326', start=0.0, duration=4.55, channel=0, text='ðàmɔ̀cò ðə̀bùlìjí ápríɲâ', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'ðɔ̀mɔ̀cɔ̀ ðə̀bùlìjí ápɾíɲá', 'gloss': 'man/male[NOM,SG] wrestle(transitive)[CLÐ,VENT,PFV] boy[ACC,SG,left_h]', 'root': 'ðɔ̀mɔ̀cɔ̀ bulij àpɾí', 'translation': 'The man wrestled the boy', 'keyword': 'ðɔ̀mɔ̀cɔ̀', 'record_type': 'positive'}, alignment=None)], features=None, recording=Recording(id='HH20210326', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH20210326.wav')], sampling_rate=16000, num_samples=50147078, duration=3134.192375, channel_ids=[0], transforms=None), custom={'fst_text': 'ðɔ̀mɔ̀cɔ̀ ðə̀bùlìjí ápɾíɲá', 'gloss': 'man/male[NOM,SG] wrestle(transitive)[CLÐ,VENT,PFV] boy[ACC,SG,left_

In [35]:
b1_kw1

{'inputs': tensor([[[-10.7486, -10.4396, -12.8877,  ...,  -8.2445,  -8.9921,  -9.2386],
          [ -9.5772,  -9.2137, -10.9537,  ...,  -8.9001,  -8.4256,  -8.5682],
          [-12.3814, -12.4497, -11.7522,  ...,  -9.4800,  -8.1260,  -8.6954],
          ...,
          [-10.9179, -10.0723, -10.3508,  ...,  -7.8718,  -7.3283,  -8.4947],
          [-12.1710, -10.5482, -11.1349,  ...,  -7.9344,  -6.4140,  -7.5113],
          [ -8.8863,  -8.6213,  -9.9693,  ...,  -6.9990,  -6.9978,  -6.8353]],
 
         [[-10.3272,  -9.8088, -11.4911,  ...,  -8.4025,  -9.0731,  -9.0645],
          [-10.1975,  -9.2222,  -9.9702,  ...,  -8.8644,  -8.7074,  -8.9946],
          [ -9.3872,  -9.7180, -15.2682,  ...,  -8.5457,  -8.7382,  -9.0556],
          ...,
          [-23.0259, -23.0259, -23.0259,  ..., -23.0259, -23.0259, -23.0259],
          [-23.0259, -23.0259, -23.0259,  ..., -23.0259, -23.0259, -23.0259],
          [-23.0259, -23.0259, -23.0259,  ..., -23.0259, -23.0259, -23.0259]]]),
 'supervisions': {

In [36]:
# encoding every keyword from the first batch
# first batch here is b1_kw1
# so we expect only one keyword to be added to the list of encoded batch labels

# batch labels
encoded_batch_labels = []

# iterate through the records
cuts_b1_kw1 = b1_kw1['supervisions']['cut']

for i in range(0, len(cuts_b1_kw1)):
    cut_meta = cuts_b1_kw1[i].custom
    kw = cut_meta['keyword']

    
    encoded_kw = tokenizer.encode(kw)
    # check if encoded keyword already in batch labels
    if encoded_kw not in encoded_batch_labels:
        print(encoded_kw)
        encoded_batch_labels.append(encoded_kw)

encoded_batch_labels    

[3, 32, 45, 2, 16, 45, 2, 6, 45, 2]


[[3, 32, 45, 2, 16, 45, 2, 6, 45, 2]]

## CTC loss
CTC loss is equivalent to the probability of a label given the logits.
For KWS inference we want to get the probability of a label occurring *within* audio (along other speech), so this isn't quite what we're looking for but it's a good first step.
We'll this approach working before we think about how to get the KWS probability.

To get CTC loss, we need the label posteriors, targets(=labels), input lengths and target lengths.
To convert logits to probabilities, use softmax.(https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html#torch.nn.functional.log_softmax).


See the [PyTorch documentation on CTC loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.ctc_loss.html#torch.nn.functional.ctc_loss) for more info.

In [32]:
output_lengths.shape, ctc_logits.shape

(torch.Size([32]), torch.Size([32, 81, 127]))

In [38]:
from torch.nn.functional import log_softmax
from torch.nn import CTCLoss

ctc_loss = CTCLoss(reduction='none')

batch_size = ctc_logits.shape[0]

# replace with the actual encoded keywords from the batch computed above
encoded_keyword = tokenizer.encode('kúkùŋú')
targets = torch.tensor([encoded_keyword]*batch_size)
target_lengths = torch.tensor([len(encoded_keyword)]*batch_size)

# TODO: cast logits into probabilities
ctc_probs = ctc_logits # your code here

loss = ctc_loss(
    # permute from batch_size, length, vocab_size -> length, batch_size, vocab_size
    # to match the input shape expected by CTC loss
    ctc_probs.permute(1, 0, 2),
    targets,
    output_lengths,
    target_lengths,
)
loss, loss.shape

(tensor([1279.0472], device='cuda:0'), torch.Size([1]))

In [39]:
output_lengths_b1.shape, ctc_logits_b1.shape

(torch.Size([2]), torch.Size([2, 224, 127]))

In [40]:
ctc_logits_b1

tensor([[[-5.5556, -6.8763, -4.8938,  ..., -3.1815, -5.8535, -4.1924],
         [-6.0214, -5.8388, -5.4177,  ..., -3.0388, -5.6230, -4.7640],
         [-6.4687, -7.9059, -5.2353,  ..., -2.8855, -5.4389, -3.6906],
         ...,
         [-6.3438, -6.2831, -6.1981,  ..., -1.8083, -6.8175, -3.9426],
         [-6.6050, -6.7224, -6.0804,  ..., -2.2763, -6.9059, -4.2840],
         [-6.0862, -6.2224, -5.1523,  ..., -2.8058, -7.0136, -4.5326]],

        [[-6.9664, -6.7803, -4.8715,  ..., -2.6160, -5.9525, -4.5607],
         [-5.6025, -6.8673, -4.4569,  ..., -2.4774, -5.8916, -5.8799],
         [-7.0487, -6.3317, -5.3575,  ..., -2.3437, -5.1800, -4.9175],
         ...,
         [-5.2646, -5.0729, -4.6845,  ..., -2.4673, -5.4221, -5.7020],
         [-5.7730, -5.5796, -4.7890,  ..., -2.6806, -6.3098, -5.8560],
         [-6.9168, -5.4737, -4.6136,  ..., -3.1816, -6.0103, -6.2778]]],
       device='cuda:0')

In [41]:
ctc_probs_test = log_softmax(ctc_logits_b1)
ctc_probs_test

tensor([[[-0.2183, -0.7423, -0.7043,  ..., -1.0153, -0.6449, -0.5259],
         [-0.9244, -0.3057, -1.2847,  ..., -1.0127, -0.5678, -0.2834],
         [-0.4446, -1.7625, -0.6339,  ..., -1.0003, -0.8310, -0.2571],
         ...,
         [-1.3717, -1.4711, -1.7126,  ..., -0.4170, -1.6168, -0.1588],
         [-1.1932, -1.4196, -1.5342,  ..., -0.5113, -1.0350, -0.1887],
         [-0.3617, -1.1360, -0.9983,  ..., -0.5228, -1.3157, -0.1609]],

        [[-1.6291, -0.6463, -0.6821,  ..., -0.4499, -0.7439, -0.8942],
         [-0.5055, -1.3342, -0.3240,  ..., -0.4513, -0.8364, -1.3993],
         [-1.0246, -0.1883, -0.7561,  ..., -0.4585, -0.5720, -1.4840],
         ...,
         [-0.2926, -0.2609, -0.1989,  ..., -1.0760, -0.2213, -1.9183],
         [-0.3613, -0.2768, -0.2429,  ..., -0.9156, -0.4389, -1.7606],
         [-1.1923, -0.3873, -0.4596,  ..., -0.8986, -0.3124, -1.9061]]],
       device='cuda:0')

In [38]:
# replace with the actual encoded keywords from the batch computed above

from torch.nn.functional import sigmoid

# batch size for kw 1 batch 1
batch_size_b1 = ctc_logits_b1.shape[0]

# encoded keyword from first batch, which is just one keyword since we're going for batches in keywords
# ðɔ̀mɔ̀cɔ̀
encoded_kw_1 = encoded_batch_labels[0]

targets_b1 = torch.tensor([encoded_kw_1]*batch_size_b1)
target_lengths_b1 = torch.tensor([len(encoded_kw_1)]*batch_size_b1)

# TODO: cast logits into probabilities
ctc_probs_b1 = log_softmax(ctc_logits_b1)

loss_b1 = ctc_loss(
    # permute from batch_size, length, vocab_size -> length, batch_size, vocab_size
    # to match the input shape expected by CTC loss
    ctc_probs_b1.permute(1, 0, 2),
    targets_b1,
    output_lengths_b1,
    target_lengths_b1,
)
loss_b1, loss_b1.shape

probs = sigmoid(loss_b1)
probs

NameError: name 'encoded_batch_labels' is not defined

## Label probability vs keyword probability
So now we've got the loss, which is equivalent to the probability of the label given the audio.
But this isn't enough for KWS.
We're not computing the probability that the audio *is* the keyword but that the audio *contains* the keyword.
To do this, we change the label so that any speech **OR** silence can be matched before or after the keyword.
This is the keyword/background model from last Fall.

How does CTC loss relate to keyword probability, and what is missing to convert it to KWS probability?
Hint: CTC loss is calculated by the intersection of two WFST graphs.
What are those graphs and what do they represent?
When doing KWS, how do the graphs differ?
I'd suggest drawing the graphs, and their resulting intersection, out using pen and paper.

Refer to [Xi et al 2025](10.1109/ICASSP49660.2025.10889332) for more explanation of KWS graph decoding.